# 01 — PyTorch Fundamentals

> **Stage 1 (Baselines)** — first executable notebook of the repo.

We load a HuggingFace model, generate sentence embeddings, and compute cosine similarity manually with PyTorch. This is the smoke test of the stack: if everything here runs cleanly, every embedder we benchmark in this repo (OpenAI, Voyage, BGE-M3, Jina/Qwen3, fine-tuned BGE-M3) is just a swap of the model id.

**What we cover**

1. Tensor basics + device selection (MPS / CUDA / CPU)
2. Loading a tokenizer + model with `transformers`
3. Inference with `torch.no_grad()` + `model.eval()`
4. Mean pooling with attention mask
5. L2 normalization + manual cosine similarity
6. The MPS warmup gotcha (first inference is 5–10× slower)

**Smoke-test model**: `BAAI/bge-small-en-v1.5` (33M params, ~130 MB download). Lightweight on purpose — the real Stage 1 baselines (OpenAI `text-embedding-3-large`, BGE-M3) come in later notebooks. The recipe (tokenize → forward → mean pool → L2 norm) transfers directly.

## 1. Imports

Three libraries: `torch` for tensors, `torch.nn.functional` for `normalize`, and `transformers` for the auto-classes. `time` is for the warmup demo at the end.

In [ ]:
import time  # For inference latency measurement in section 9 (MPS warmup demo).

# torch: PyTorch's main library — tensors and vectorized ops (the backbone).
import torch
# torch.nn.functional: stateless math ops (activations, normalize, softmax).
# Used here for F.normalize in section 8.
import torch.nn.functional as F
# AutoModel + AutoTokenizer: HuggingFace infers the right architecture from the model's config.
# Same API works for BERT, RoBERTa, BGE, Qwen — only the model_id changes.
from transformers import AutoModel, AutoTokenizer

# Print PyTorch version + which hardware backends are available.
#   torch.backends.mps → Metal Performance Shaders (Apple Silicon GPU: M1/M2/M3/M4).
#   torch.cuda         → NVIDIA GPU. Not present on this Mac, expect False.
print(f"PyTorch       : {torch.__version__}")
print(f"MPS available : {torch.backends.mps.is_available()}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Device detection

We pick **MPS** if running on Apple Silicon, **CUDA** if running on a NVIDIA box, otherwise **CPU**. The order matters: in a Mac with both available (rare), we still want MPS — it's the GPU on the user's machine.

> **Heads up**: MPS doesn't support `float64`. PyTorch defaults to `float32` on MPS, so this is rarely a problem in practice — but if you load NumPy arrays with `dtype=float64` and move them to MPS, cast first.

In [ ]:
def get_device() -> str:
    """Return the best available device, priority: MPS > CUDA > CPU."""
    # MPS first: on Apple Silicon, we want the integrated GPU.
    # PyTorch JIT-compiles Metal kernels on first call (see warmup gotcha in section 9).
    if torch.backends.mps.is_available():
        return "mps"
    # CUDA second: NVIDIA GPU, the standard choice on ML servers.
    if torch.cuda.is_available():
        return "cuda"
    # CPU as fallback: works on any machine, but ~10-100× slower than GPU.
    return "cpu"


# Call the helper ONCE and store the result in a global constant.
# The rest of the notebook uses this variable to move tensors to the right device.
device = get_device()
print(f"Using device: {device}")

## 3. Tensor basics

A `torch.Tensor` is like a NumPy `ndarray` with three superpowers: it can live on a GPU, it tracks gradients (autograd, opt-in), and it integrates with PyTorch's neural network layers. Same indexing, same broadcasting, same vectorized ops — if you know NumPy, you know 90% of tensors.

In [ ]:
# Create a 1D tensor of 3 floats — PyTorch's "hello world".
# torch.tensor() infers dtype automatically: Python floats → torch.float32.
x = torch.tensor([1.0, 2.0, 3.0])

# Key attributes of any Tensor:
#   .shape  → tuple of dimensions (here: torch.Size([3]) = vector of 3 elements)
#   .dtype  → numeric type (float32 is the default; MPS doesn't support float64)
#   .device → where the tensor lives in memory ('cpu', 'mps', 'cuda:0', etc.)
print(f"x        = {x}")
print(f"x.shape  = {x.shape}")
print(f"x.dtype  = {x.dtype}")
print(f"x.device = {x.device}")

# .to(device) moves the tensor to the selected device.
# IMPORTANT: this does NOT modify the original — it returns a COPY on the new device.
# That's why the reassignment to x_dev is necessary.
x_dev = x.to(device)
print(f"\nAfter .to({device!r}):")
print(f"x_dev.device = {x_dev.device}")

## 4. Load tokenizer + model

We use `BAAI/bge-small-en-v1.5` as a smoke test:

- 33M params, ~130 MB download — the first run pulls weights from the Hub; subsequent runs read from `~/.cache/huggingface/hub/`.
- Same tokenizer family and pooling recipe as BGE-M3 (used later in this Stage), so what you learn here transfers.
- This is **not** the embedder we'll benchmark — it's a toy choice to validate the stack. The real Stage 1 baselines come in `02_openai_baseline.ipynb` and `03_bge_m3_baseline.ipynb`.

> **Gotcha — `HF_TOKEN` warning**: `transformers` will print a warning about `HF_TOKEN` not being set. It's defensive — public models like this one don't need a token. Ignore it. The warning matters only for gated or private repos.

In [ ]:
# Model identifier on the HuggingFace Hub: <org>/<model-name>.
# On the first run, transformers downloads the weights (~130 MB) and caches them
# in ~/.cache/huggingface/hub/. Subsequent runs are instant.
MODEL_ID = "BAAI/bge-small-en-v1.5"

# AutoTokenizer infers the right tokenizer class from the model's config.
# BGE-small uses WordPiece (inherited from BERT) — splits words into subword tokens.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Chain three operations in one line:
#   .from_pretrained(MODEL_ID) → download + load weights + architecture from the Hub
#   .to(device)                → move ALL weights to the device (MPS/CUDA/CPU)
#   .eval()                    → eval mode: disable dropout, batch-norm uses running stats
# .eval() is CRITICAL at inference time — without it, dropout introduces randomness
# and embeddings change between identical runs.
model = AutoModel.from_pretrained(MODEL_ID).to(device).eval()

# Useful model metadata:
#   hidden_size             → dimensionality of each embedding vector (384 for BGE-small)
#   max_position_embeddings → max input length in tokens (512 — anything longer is truncated)
#   vocab_size              → unique tokens the tokenizer knows (30,522 for BGE-small)
print(f"Model: {MODEL_ID}")
print(f"Hidden size            : {model.config.hidden_size}")
print(f"Max position embeddings: {model.config.max_position_embeddings}")
print(f"Vocab size             : {tokenizer.vocab_size}")

## 5. Tokenize a batch

Three sentences: two are semantically related (Apple earnings), the third is unrelated. The tokenizer:

- splits each sentence into subword tokens
- pads shorter sequences to match the longest in the batch (`padding=True`)
- truncates anything past the model's max length (`truncation=True`)
- returns PyTorch tensors directly (`return_tensors="pt"`)

The `attention_mask` is the bookkeeping that tells the model (and us, later) which tokens are real and which are padding.

In [ ]:
# Batch of 3 sentences designed to validate that the embedder works:
#   - The first two are about Apple/earnings → should have high similarity to each other
#   - The third is unrelated → should have low similarity to the first two
# If the model embeds these 3 and the similarity matrix matches that pattern, the stack is OK.
sentences = [
    "Apple reported record revenue in fiscal Q4.",
    "iPhone sales drove Apple's quarterly earnings.",
    "The cat sat on the mat.",
]

# The tokenizer does 4 things in one call:
#   padding=True        → adds [PAD] tokens to shorter sentences so ALL have the same
#                         length (the longest in the batch). Required because PyTorch
#                         tensors need uniform shape.
#   truncation=True     → cuts any sentence longer than max_position_embeddings (512).
#   return_tensors="pt" → returns PyTorch tensors (instead of Python lists or numpy).
# The .to(device) at the end moves the batch (input_ids + attention_mask) to the model's device.
# CRITICAL: model and data must be on the SAME device, otherwise PyTorch raises an error.
batch = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    return_tensors="pt",
).to(device)

# The tokenizer returns a dict with 2 key tensors:
#   input_ids      → numeric IDs for each token (e.g., 6207 = "apple", 2988 = "reported")
#   attention_mask → 1 = real token, 0 = padding. The model uses it to ignore padding.
# Both have shape (batch, seq_len). In this case: 3 sentences × 11 tokens (the longest one).
print(f"input_ids.shape      = {tuple(batch['input_ids'].shape)}      # (batch, seq_len)")
print(f"attention_mask.shape = {tuple(batch['attention_mask'].shape)} # (batch, seq_len)")

# Print the first 12 tokens of the first sentence to inspect:
#   input_ids: starts with 101 (special [CLS] token) and ends with 102 ([SEP]).
#   attention_mask: all 1s because there's no padding in this sentence (it's the longest).
print(f"\nFirst sentence input_ids[:12]      = {batch['input_ids'][0, :12].tolist()}")
print(f"First sentence attention_mask[:12] = {batch['attention_mask'][0, :12].tolist()}")

## 6. Inference with `torch.no_grad()`

Two flags matter for inference, and they are **not the same**:

| Flag | What it does | When |
|---|---|---|
| `model.eval()` | Disables dropout and switches batch-norm to running stats | Set once after loading |
| `torch.no_grad()` | Disables autograd graph construction | Wrap each inference call |

We already called `model.eval()` when loading. Now we wrap the forward pass in `torch.no_grad()` — less memory, faster, no risk of accidentally building a gradient graph for tensors we don't plan to backprop through.

In [ ]:
# torch.no_grad() is a context manager that DISABLES gradient tracking (autograd)
# for everything inside it. Benefits at inference time:
#   1. Less memory  — no computational graph built for backprop
#   2. Faster       — skips internal bookkeeping operations
#   3. Safer        — impossible to accidentally create gradients that break things later
# IMPORTANT: model.eval() (set earlier) and torch.no_grad() are DIFFERENT things.
# eval() affects dropout/batch-norm; no_grad() affects the autograd graph. You need both.
with torch.no_grad():
    # **batch unpacks the dict as kwargs:
    #   model(input_ids=..., attention_mask=...)
    # Forward pass: tokens → contextual embeddings per layer → final layer hidden state.
    outputs = model(**batch)

# outputs is a BaseModelOutput object with several optional attributes.
# last_hidden_state is the most-used one: the output of the last Transformer layer,
# ONE VECTOR PER TOKEN. Shape: (batch, seq_len, hidden_dim) = (3, 11, 384).
# To get ONE vector PER SENTENCE, we need to pool (next cell).
last_hidden = outputs.last_hidden_state
print(f"last_hidden.shape = {tuple(last_hidden.shape)}  # (batch, seq_len, hidden_dim)")

## 7. Mean pooling

`last_hidden_state` has one vector per token. To get one vector per sentence, we average across the `seq_len` dimension — but we use `attention_mask` to ignore padding tokens. If we just averaged blindly, padding positions (which carry near-zero contextual signal) would dilute the real content.

> **Why mean pooling and not CLS pooling?** Different models train with different recipes. BGE-small uses mean pooling. BGE-M3 (next notebook) uses CLS for the dense vector — always check the model card. The mean-pooling helper below is reused as-is for any model that documents mean pooling.

In [ ]:
def mean_pooling(
    token_embeddings: torch.Tensor,   # (batch, seq_len, hidden_dim) — one vector per token
    attention_mask: torch.Tensor,     # (batch, seq_len) — 1 = real token, 0 = padding
) -> torch.Tensor:
    """Average token embeddings along seq_len, ignoring padding via attention_mask."""

    # Expand the mask from (batch, seq_len) to (batch, seq_len, hidden_dim)
    # so we can multiply it element-wise against the embeddings.
    #   .unsqueeze(-1)  → adds a final dim: (batch, seq_len, 1)
    #   .expand(...)    → replicates that dim hidden_dim times WITHOUT copying memory (broadcast view)
    #   .float()        → cast to float so multiplication matches the embeddings' dtype
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    # Multiply tokens × mask: padding tokens get zeroed out (× 0).
    # Sum along seq_len (dim=1) → (batch, hidden_dim).
    # Each row is the sum of embeddings of ALL real tokens in that sentence.
    summed = (token_embeddings * mask).sum(dim=1)

    # Count how many real tokens per sentence (sum the mask).
    # .clamp(min=1e-9) protects against division-by-zero in the degenerate case
    # of an all-padding sentence (shouldn't happen, but cheap defensive guard).
    counts = mask.sum(dim=1).clamp(min=1e-9)

    # Element-wise division: sum / count = average (mean pooling).
    # Final result: (batch, hidden_dim) — one vector per sentence, ready to normalize.
    return summed / counts


# Apply the helper to the model's output: pass per-token embeddings and the mask.
embeddings = mean_pooling(last_hidden, batch["attention_mask"])
print(f"embeddings.shape = {tuple(embeddings.shape)}  # (batch, hidden_dim)")

## 8. L2 normalization + cosine similarity

Cosine similarity is `(a · b) / (‖a‖ · ‖b‖)`. If we normalize `a` and `b` to unit length, the denominators collapse to 1 and the formula reduces to a plain dot product — faster in batch, mathematically identical, and the result lives in `[-1, 1]`.

This is the same identity that powers retrieval at scale: vector databases store **normalized** vectors so that nearest-neighbor search by dot product is equivalent to nearest-neighbor by cosine similarity.

After normalization we expect each row of `embeddings` to have norm 1. We verify, then compute the full pairwise similarity matrix with a single matmul.

In [ ]:
# F.normalize divides each vector by its norm → unit-length vectors.
#   p=2   → L2 norm (Euclidean): sqrt(sum(x_i^2))
#   dim=1 → normalize along hidden_dim (each row is a sentence)
# Result: each row has magnitude 1.0, preserving the original direction.
# This turns cosine similarity into a plain dot product (mathematically equivalent).
embeddings = F.normalize(embeddings, p=2, dim=1)

# Sanity check: norms should be 1.0 (with negligible floating-point error).
# If any isn't 1.0, something went wrong in normalize and it's debugging time.
print(f"Norms after normalize: {[round(n, 4) for n in embeddings.norm(dim=1).tolist()]}")

# Dot product between ALL pairs of embeddings = cosine similarity matrix.
#   embeddings.shape    = (3, 384)
#   embeddings.T.shape  = (384, 3)
#   embeddings @ ...    = (3, 3) → symmetric matrix with diagonal = 1.0
# The @ operator is matmul (matrix multiplication). Equivalent to torch.matmul().
sim_matrix = embeddings @ embeddings.T

# .cpu()    → move tensor from MPS/CUDA to CPU (numpy/print only work on CPU).
# .numpy()  → convert tensor to ndarray for cleaner printing.
# .round(3) → round to 3 decimals so the matrix is readable.
print("\nCosine similarity matrix:")
print(sim_matrix.cpu().numpy().round(3))

**Reading the matrix**: the diagonal is always 1.0 (each sentence vs itself). Off-diagonal entries `[0,1]` and `[1,0]` should be high — both sentences are about Apple's earnings — while everything involving sentence 2 ("the cat...") should be markedly lower. If the matrix matches that pattern, the embedder is doing what it's supposed to.

## 9. The MPS warmup gotcha

The very first inference call on MPS triggers JIT compilation of Metal kernels — it can be 5–10× slower than subsequent runs. The same effect exists on CUDA but is much less pronounced.

**Always discard the first measurement** when timing inference on MPS. It's the kernel cache warming up, not your machine being slow. We synchronize the device before reading the clock so we're measuring real GPU work, not just kernel dispatch.

In [ ]:
def time_inference() -> float:
    """Return milliseconds for ONE forward pass over the current batch."""
    # time.perf_counter() is Python's highest-resolution clock (microseconds).
    # Better than time.time() for benchmarking, which has millisecond resolution.
    start = time.perf_counter()

    # Forward pass without gradients (same as section 6).
    # The _ means we discard the output — we only measure time, the result is unused.
    with torch.no_grad():
        _ = model(**batch)

    # CRITICAL: GPU operations are ASYNCHRONOUS by default.
    # When model(**batch) "returns", PyTorch has only enqueued the kernels — the GPU
    # may still be working. If we measured before syncing, we'd time only the
    # dispatch (microseconds), not the real work (milliseconds).
    # synchronize() blocks until the GPU finishes ALL enqueued work.
    if device == "mps":
        torch.mps.synchronize()
    elif device == "cuda":
        torch.cuda.synchronize()
    # On CPU we don't need to synchronize — execution is already synchronous.

    # Convert seconds → milliseconds (more readable for inference).
    return (time.perf_counter() - start) * 1000


# Run 5 consecutive measurements to see the MPS warmup effect:
#   Run 1:    slow (5-10× more) — JIT-compiles Metal kernels for the first time.
#   Run 2-5:  fast (warm state) — kernels already in cache.
# Lesson: ALWAYS discard the first measurement when benchmarking on MPS/CUDA.
for i in range(1, 6):
    label = "warmup, ignore" if i == 1 else ""
    print(f"Run {i}: {time_inference():6.1f} ms  {label}")

## What's next

You now have every primitive needed to build the embedder wrappers used across this repo:

| Concept | Reused in |
|---|---|
| `AutoTokenizer` + `AutoModel` | All HuggingFace embedders (`src/embeddings/`) |
| Mean pooling + L2 norm | BGE-small here, BGE-M3 in `03_bge_m3_baseline` |
| `torch.no_grad()` + `model.eval()` | Every embedder wrapper |
| Cosine similarity matrix | Retrieval scoring across all baselines |

Next notebook: load the FinanceBench dataset and turn 10-K filings into chunks ready to embed (`paso-03` in the course, `02_financebench_loader` here).